# 11 — Extension modelling

This final model extension provides a focused example of more purposeful treatment of feature inputs. It reflects only a small part of the broader model-development evolution that occurred beyond the scope of this repository.

The original `TCN_starNLL` lineage is preserved. This notebook adds one bounded extension: causal upstream estimates for demand and regional direct radiation at the forecast origin, together with timestamp-aligned estimates of donor variability. Development remains confined to 2015–2019; the frozen 2020 evaluation is not opened or used.

## Purposeful treatment of availability

The extension separates data preparation from modelling. Before any TCN sequence is constructed, the upstream pipeline creates `x_estimated` and `x_estimated_sd` for every eligible timestamp using only earlier observations. The TCN never calculates donor averages or standard deviations.

For a sequence ending at forecast origin $t$, the physical channel is $[x_{t-n},\ldots,x_{t-1},x^{est}_t]$. The complete, timestamp-aligned `x_estimated_sd` series is supplied as an additional temporal feature. The estimate and its uncertainty therefore always refer to the same time.

## Causal donor rules

### Demand

- An ordinary weekday uses the same five-minute clock block and weekday in each of the previous three weeks.
- If one of those donor dates is a Queensland public holiday, that donor is replaced by the nearest Sunday at the same clock time.
- A non-holiday weekend uses the same weekend day and clock block in each of the previous three weeks.
- A current public holiday uses the same clock block on each of the three previous Sundays.
- The mean and sample standard deviation (`ddof=1`) are emitted only when all three donors exist. Initial missing values are retained.

### Direct radiation

Each regional estimate uses same-clock observations from the previous three calendar days. The historical radiation channel uses the last completed source hour; only its terminal value is replaced by `direct_radiation_estimated[t]`. Every donor predates the timestamp being estimated.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tcn_star_nll_extension_data import load_frame, prepare_fold, fold_endpoints
from src.tcn_star_nll_extension_model import TCNStarNLLCausalFeatureEstimates

CONFIG_PATH = ROOT / 'config/experiments/tcn_star_nll_causal_feature_estimates_validate_2019_seed42.yml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config['model_name'], config['variant_name'], config['split']

## Build and audit the upstream helpers

The next cell constructs the complete helper columns before any TCN dataset exists. It verifies the usable boundary, post-boundary completeness and the 2020 data ceiling.

In [ ]:
frame = load_frame(
    ROOT / config['dataset'],
    ROOT / config['helper_sources']['demand'],
    ROOT / config['helper_sources']['direct_radiation'],
)
helper_columns = ['demand_estimated', 'demand_estimated_sd'] + [
    column for column in frame.columns if 'direct_radiation' in column
]
pd.Series({
    'first_usable_timestamp': frame.index.min(),
    'last_available_timestamp': frame.index.max(),
    'rows': len(frame),
    'helper_columns': len(helper_columns),
    'helper_missing_values': int(frame[helper_columns].isna().sum().sum()),
    'post_2020_rows': int((frame.index >= pd.Timestamp('2021-01-01')).sum()),
})

## Scaling choices

All transformations are fitted on 2015–2018 only. `demand_estimated` uses the original demand mean and standard deviation because it occupies the terminal position of that channel. `demand_estimated_sd` is divided by the demand training standard deviation without mean subtraction because dispersion has no location offset.

Regional direct-radiation observations and estimates use the same per-region training-fold minimum and maximum and are clipped to $[0,1]$. `direct_radiation_estimated_sd` is divided by the same regional range without centering. Using the physical feature's scale preserves the meaning of the uncertainty value and prevents validation information entering preprocessing.

In [ ]:
train_start = pd.Timestamp('2015-01-01')
train_end = pd.Timestamp('2019-01-01')
validation_start = pd.Timestamp('2019-01-01')
validation_end = pd.Timestamp('2020-01-01')
arrays, preprocessing = prepare_fold(frame, train_start, train_end)
train_endpoints = fold_endpoints(frame, train_start, train_end)
validation_endpoints = fold_endpoints(frame, validation_start, validation_end)
pd.Series({
    'training_origins': len(train_endpoints),
    'validation_origins': len(validation_endpoints),
    'demand_tensor_width': arrays['demand'].shape[1],
    'regional_tensor_width': arrays['regional'].shape[2],
    'calendar_encoded_width': preprocessing['calendar_encoded_width'],
})

## Model construction and justification

The probabilistic backbone, 505-state lookback, causal convolutions, dilation schedule, dropout, fusion width, six horizons and bounded Student-t heads remain unchanged. Demand widens from 7 to 8 channels for aligned `demand_estimated_sd`. Each regional branch widens from 10 to 12 channels for direct radiation and aligned `direct_radiation_estimated_sd`. The estimate itself does not create another channel: it replaces only the terminal value of its corresponding physical channel.

In [ ]:
model = TCNStarNLLCausalFeatureEstimates(calendar_dim=preprocessing['calendar_encoded_width'])
pd.Series({
    'parameters': sum(parameter.numel() for parameter in model.parameters()),
    'receptive_field_steps': model.receptive_field_steps,
    'demand_channels': model.DEMAND_CHANNELS,
    'regional_channels_per_region': model.REGIONAL_FEATURES,
})

## Exact authorized development run

The experiment trains on 2015–2018, validates on 2019 and uses seed 42. It retains AdamW, effective batch size 512, learning-rate reduction, early stopping and minimum-validation-NLL checkpoint selection. The frozen 2020 gate remains untouched. Full training is disabled by default inside the notebook to prevent accidental reruns.

In [ ]:
RUN_FULL_TRAINING = False
command = [
    sys.executable,
    str(ROOT / 'tools/run_tcn_star_nll_causal_feature_estimates_validate_2019.py'),
    '--authorise-full-run',
]
print(' '.join(command))
if RUN_FULL_TRAINING:
    import subprocess
    subprocess.run(command, cwd=ROOT, check=True)

## Retained 2019 result and interpretation boundary

The public release retains a small comparison table rather than depending on ignored checkpoints or local training logs. All three neural models were trained on 2015–2018, validated on 2019 and use seed 42. Their MAE and RMSE are taken from the checkpoint selected by minimum validation NLL. AEMO is the aligned 2019 P5MIN external point-forecast benchmark, so it has neither a training seed nor a probabilistic NLL checkpoint.

| Rank by MAE | Model | Role | Best epoch | Validation NLL | MAE (MW) | RMSE (MW) |
|---:|---|---|---:|---:|---:|---:|
| 1 | `TCN_starNLL_causal_feature_estimates_no_sd` | Extension without SD channels | 10 | 0.734999 | 38.091 | 49.973 |
| 2 | `TCN_starNLL_causal_feature_estimates` | Extension with SD channels | 11 | 0.736293 | 38.203 | 50.165 |
| 3 | `TCN_starNLL_direct_radiation` | Previous non-extension champion | 7 | 0.737551 | 38.287 | 50.301 |
| 4 | AEMO P5MIN external benchmark | External point forecast | — | — | 47.119 | 61.315 |

The no-SD extension records the best value on all three available metrics, but its advantage over the SD-inclusive extension is small. This supports the simpler specification on parsimony grounds rather than a claim of substantial improvement. The extension is not part of the original completed model-selection path, does not supersede the published `TCN_starNLL` result, and does not open the fixed 2020 chronological evaluation.

In [ ]:
comparison_path = ROOT / 'outputs/11_extension_modelling/model_comparison_2019_seed42.csv'
comparison = pd.read_csv(comparison_path, keep_default_na=False)
display(comparison)